# Brain Tumor — entraînement et évaluation sur Kaggle

Notebook autonome : il ne dépend pas du dossier `src/` du projet. Dans Kaggle, ajoute le dataset avec **Add Input**, active un GPU dans **Settings → Accelerator**, puis exécute les cellules dans l’ordre. Les données d’entrée sont en lecture seule; les modèles, graphiques et métriques seront enregistrés dans `/kaggle/working/`.

Le notebook détecte les dossiers de classes sous `/kaggle/input`. Il préserve un split Test fourni par le dataset; sinon, il crée une séparation stratifiée 70/15/15. InceptionV3 utilise les poids ImageNet, donc l’accès Internet Kaggle doit être activé au premier entraînement pour télécharger ces poids.

In [1]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras import Model, layers
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_FEATURE_EXTRACTION = 10
EPOCHS_FINE_TUNING = 10
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow :", tf.__version__)
print("GPU(s) disponibles :", tf.config.list_physical_devices("GPU"))
print("Datasets Kaggle :", list(Path("/kaggle/input").iterdir()) if Path("/kaggle/input").exists() else "/kaggle/input introuvable")
print("Résultats enregistrés dans :", OUTPUT_DIR)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
SPLIT_ALIASES = {
    "train": "train", "training": "train",
    "val": "validation", "valid": "validation", "validation": "validation",
    "test": "test", "testing": "test",
}
INPUT_DIR = Path("/kaggle/input")
if not INPUT_DIR.is_dir():
    raise FileNotFoundError("Dossier Kaggle /kaggle/input introuvable. Exécute ce notebook sur Kaggle et ajoute le dataset avec Add Input.")

valid_rows = []
invalid_files = []
for image_path in INPUT_DIR.rglob("*"):
    if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
        continue
    try:
        with Image.open(image_path) as image:
            image.verify()
        with Image.open(image_path) as image:
            image.convert("RGB").load()
    except (OSError, ValueError, UnidentifiedImageError) as error:
        invalid_files.append({"path": str(image_path), "reason": str(error)})
        continue

    parts = image_path.relative_to(INPUT_DIR).parts
    lower_parts = [part.lower() for part in parts]
    split_index = next((i for i, part in enumerate(lower_parts) if part in SPLIT_ALIASES), None)
    if "images" in lower_parts:
        image_folder_index = lower_parts.index("images")
        if image_folder_index == 0:
            invalid_files.append({"path": str(image_path), "reason": "class folder not found"})
            continue
        class_name = parts[image_folder_index - 1]
    elif split_index is not None and split_index + 1 < len(parts) - 1:
        class_name = parts[split_index + 1]
    elif len(parts) >= 2:
        class_name = parts[-2]
    else:
        invalid_files.append({"path": str(image_path), "reason": "class folder not found"})
        continue

    split_name = SPLIT_ALIASES.get(lower_parts[split_index]) if split_index is not None else None
    valid_rows.append({"path": str(image_path), "label": class_name, "source_split": split_name})

images = pd.DataFrame(valid_rows, columns=["path", "label", "source_split"])
if images.empty:
    raise ValueError("Aucune image de classe trouvée. Vérifie l’arborescence du dataset sous /kaggle/input.")
print(f"Images valides : {len(images)} | images/fichiers illisibles ignorés : {len(invalid_files)}")
print("Classes détectées :", sorted(images["label"].unique()))
display(images.groupby(["source_split", "label"], dropna=False).size().rename("images").to_frame())
if invalid_files:
    display(pd.DataFrame(invalid_files).head(10))

has_test = (images["source_split"] == "test").any()
if has_test:
    test_df = images[images["source_split"] == "test"].copy()
    available_train = images[images["source_split"] != "test"].copy()
    if available_train.empty:
        raise ValueError("Le dataset fournit Test mais aucune image pour entraîner le modèle.")
    has_validation = (available_train["source_split"] == "validation").any()
    if has_validation:
        train_df = available_train[available_train["source_split"] == "train"].copy()
        val_df = available_train[available_train["source_split"] == "validation"].copy()
        if train_df.empty:
            train_df = available_train[available_train["source_split"] != "validation"].copy()
    else:
        train_df, val_df = train_test_split(
            available_train,
            test_size=0.15,
            random_state=SEED,
            stratify=available_train["label"],
        )
else:
    train_val_df, test_df = train_test_split(
        images,
        test_size=0.15,
        random_state=SEED,
        stratify=images["label"],
    )
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=0.15 / 0.85,
        random_state=SEED,
        stratify=train_val_df["label"],
    )

class_names = sorted(images["label"].unique().tolist())
class_to_index = {name: index for index, name in enumerate(class_names)}
for name, frame in (("Train", train_df), ("Validation", val_df), ("Test", test_df)):
    missing = sorted(set(class_names) - set(frame["label"].unique()))
    if missing:
        raise ValueError(f"{name} n’a pas d’image pour ces classes : {missing}. Vérifie les splits du dataset.")
    print(f"{name}: {len(frame)} images | {frame['label'].value_counts().to_dict()}")

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()
for frame in (train_df, val_df, test_df):
    frame["class_id"] = frame["label"].map(class_to_index).astype("int32")

(OUTPUT_DIR / "class_names.json").write_text(
    json.dumps(class_names, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Ordre classes/indices :", class_to_index)

In [ ]:
def oversample_training(frame):
    counts = frame["label"].value_counts()
    target_count = int(counts.max())
    balanced_parts = []
    for class_name in class_names:
        class_rows = frame[frame["label"] == class_name]
        balanced_parts.append(
            class_rows.sample(n=target_count, replace=len(class_rows) < target_count, random_state=SEED)
        )
    return pd.concat(balanced_parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

balanced_train_df = oversample_training(train_df)
print("Train original :", train_df["label"].value_counts().to_dict())
print("Train équilibré :", balanced_train_df["label"].value_counts().to_dict())

def augment_train_image(image):
    image = tf.image.random_flip_left_right(image)
    image = image / 255.0
    channels = tf.shape(image)[-1]
    blur_shape = tf.stack([3, 3, channels, 1])
    blur_kernel = tf.ones(blur_shape, dtype=image.dtype) / 9.0
    image = tf.cond(
        tf.random.uniform(()) < 0.3,
        lambda: tf.nn.depthwise_conv2d(
            image[tf.newaxis, ...], blur_kernel, strides=[1, 1, 1, 1], padding="SAME"
        )[0],
        lambda: image,
    )
    image = tf.clip_by_value(image + tf.random.normal(tf.shape(image), stddev=0.03), 0.0, 1.0)
    return image * 255.0

def make_dataset(frame, training=False):
    paths = frame["path"].astype(str).to_numpy()
    labels = frame["class_id"].to_numpy(dtype=np.int32)
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)

    def load_image(path, label):
        image_bytes = tf.io.read_file(path)
        image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMAGE_SIZE)
        if training:
            image = augment_train_image(image)
        image = tf.keras.applications.inception_v3.preprocess_input(tf.cast(image, tf.float32))
        return image, label

    return (
        dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

train_ds = make_dataset(balanced_train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

sample_images, sample_labels = next(iter(train_ds))
print("Batch Train :", sample_images.shape, sample_images.dtype)
print("Labels batch :", sample_labels[:10].numpy())
print("Normalisation InceptionV3 : environ [-1, 1]. Flou, bruit et flip : Train uniquement.")

## Phase 1 — Feature extraction

On entraîne la tête de classification pendant que la base InceptionV3 pré-entraînée reste gelée.

In [ ]:
base_model = InceptionV3(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMAGE_SIZE, 3),
)
base_model.trainable = False
inputs = layers.Input(shape=(*IMAGE_SIZE, 3))
features = base_model(inputs, training=False)
features = layers.GlobalAveragePooling2D()(features)
features = layers.Dropout(0.3)(features)
outputs = layers.Dense(len(class_names), activation="softmax")(features)
model = Model(inputs, outputs, name="brain_tumor_inceptionv3")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

phase1_path = OUTPUT_DIR / "phase1_best.keras"
phase1_callbacks = [
    ModelCheckpoint(phase1_path, monitor="val_accuracy", mode="max", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
]
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FEATURE_EXTRACTION,
    callbacks=phase1_callbacks,
)

## Phase 2 — Fine-tuning

On dégèle les dernières couches convolutionnelles. Les couches BatchNormalization restent gelées pour éviter de déstabiliser leurs statistiques pré-entraînées.

In [ ]:
base_model.trainable = True
unfreeze_from_layer = 249
for layer_index, layer in enumerate(base_model.layers):
    layer.trainable = layer_index >= unfreeze_from_layer and not isinstance(layer, layers.BatchNormalization)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
phase2_path = OUTPUT_DIR / "phase2_best.keras"
phase2_callbacks = [
    ModelCheckpoint(phase2_path, monitor="val_accuracy", mode="max", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
]
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE_TUNING,
    callbacks=phase2_callbacks,
)

phase1_best_val_accuracy = max(history_phase1.history["val_accuracy"])
phase2_best_val_accuracy = max(history_phase2.history["val_accuracy"])
selected_model_path = phase2_path if phase2_best_val_accuracy >= phase1_best_val_accuracy else phase1_path
model = tf.keras.models.load_model(selected_model_path)
final_model_path = OUTPUT_DIR / "brain_tumor_inceptionv3.keras"
model.save(final_model_path)
print(f"Meilleure validation phase 1 : {phase1_best_val_accuracy:.4f}")
print(f"Meilleure validation phase 2 : {phase2_best_val_accuracy:.4f}")
print("Checkpoint choisi :", selected_model_path.name)
print("Modèle final sauvegardé :", final_model_path)

## Évaluation finale sur Test

Cette partie est exécutée après le choix du checkpoint sur Validation. Elle calcule les métriques et le rapport par classe, trace la matrice de confusion et les courbes, puis enregistre tous les résultats.

In [ ]:
test_metrics = model.evaluate(test_ds, verbose=1, return_dict=True)
print("Métriques Test :")
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
y_probabilities = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_probabilities, axis=1)
report = classification_report(
    y_true,
    y_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
display(pd.DataFrame(report).transpose().round(4))

cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title("Matrice de confusion — Test")
axes[0].set_xlabel("Prédit")
axes[0].set_ylabel("Réel")

phase1_history = history_phase1.history
phase2_history = history_phase2.history
train_accuracy = phase1_history.get("accuracy", []) + phase2_history.get("accuracy", [])
val_accuracy = phase1_history.get("val_accuracy", []) + phase2_history.get("val_accuracy", [])
train_loss = phase1_history.get("loss", []) + phase2_history.get("loss", [])
val_loss = phase1_history.get("val_loss", []) + phase2_history.get("val_loss", [])
epochs = np.arange(1, len(train_accuracy) + 1)
phase_boundary = len(phase1_history.get("accuracy", [])) + 0.5
axes[1].plot(epochs, train_accuracy, label="Train")
axes[1].plot(epochs, val_accuracy, label="Validation")
axes[1].axvline(phase_boundary, color="gray", linestyle="--")
axes[1].set_title("Accuracy par epoch")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[2].plot(epochs, train_loss, label="Train")
axes[2].plot(epochs, val_loss, label="Validation")
axes[2].axvline(phase_boundary, color="gray", linestyle="--")
axes[2].set_title("Loss par epoch")
axes[2].set_xlabel("Epoch")
axes[2].legend()
plt.tight_layout()

confusion_path = OUTPUT_DIR / "confusion_matrix_test.png"
curves_path = OUTPUT_DIR / "training_curves.png"
metrics_path = OUTPUT_DIR / "test_metrics.json"
metrics_payload = {
    "test_metrics": {key: float(value) for key, value in test_metrics.items()},
    "classification_report": report,
    "class_names": class_names,
    "selected_checkpoint": selected_model_path.name,
}
metrics_path.write_text(json.dumps(metrics_payload, ensure_ascii=False, indent=2), encoding="utf-8")
fig.savefig(confusion_path, dpi=150, bbox_inches="tight")
plt.close(fig)
fig_curves, curve_axes = plt.subplots(1, 2, figsize=(13, 5))
curve_axes[0].plot(epochs, train_accuracy, label="Train")
curve_axes[0].plot(epochs, val_accuracy, label="Validation")
curve_axes[0].axvline(phase_boundary, color="gray", linestyle="--", label="Début fine-tuning")
curve_axes[0].set_title("Accuracy")
curve_axes[0].set_xlabel("Epoch")
curve_axes[0].legend()
curve_axes[1].plot(epochs, train_loss, label="Train")
curve_axes[1].plot(epochs, val_loss, label="Validation")
curve_axes[1].axvline(phase_boundary, color="gray", linestyle="--", label="Début fine-tuning")
curve_axes[1].set_title("Loss")
curve_axes[1].set_xlabel("Epoch")
curve_axes[1].legend()
fig_curves.tight_layout()
fig_curves.savefig(curves_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig_curves)

print("Modèle :", final_model_path)
print("Noms de classes :", OUTPUT_DIR / "class_names.json")
print("Métriques :", metrics_path)
print("Matrice de confusion :", confusion_path)
print("Courbes :", curves_path)
print("Dans Kaggle, télécharge ces fichiers depuis l’onglet Output de la session.")